In [100]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import CubicSpline

J = 1
# constante de acoplamento
h = 0
# campo magnético externo
kb = 1
# constante de boltzmman
rnd = np.random.default_rng(seed=1)
# gerador de número aleatórios


# Classe do Model de Ising
class IsingModel:

    def __init__(self, L, T):
        # parametros do modelo
        self.L = L
        self.T = T

        # array com os spins
        self.spins = np.ones((L, L))

        # arrays para guardar evolução das variáveis
        self.energies = []
        self.magnetizations = []

    def calc_ener_spin(self, i, j)->float:
        viz = self.neighbours((i, j))
        own = self.spins[i][j]
        energy = -1 * J * own * np.cumsum(self.spins * viz)[-1]
        return energy

    def calc_ener(self)->float:
        # para não termos de tar a complicar na escolha dos vizinhos podemos calcular todas as interações duas vezes e dividir por dois
        energy = 0
        for i in range(self.L):
            for j in range(self.L):
                energy += 0.5 * self.calc_ener_spin(i, j)
        energy += -1 * h * np.cumsum(self.spins)[-1] 

        return energy

    def calc_mag(self)->float:
        # calcular a magnetização por spin do sistema
        M = J * np.cumsum(self.spins)[-1]
        mag = M / self.L**2
        return mag
    
    def flip(self, i, j)->None:
        self.spins[i][j] *= -1 
    
    def iter_monte_carlo(self, n_iter):
        # iterar com o método de Metropolis Hastings
        Ei = self.calc_ener()
        self.energies.append(Ei)

        mag = self.calc_mag()
        self.magnetizations.append(mag)

        for i in tqdm(range(n_iter), desc=f"L={self.L:6d}, T={self.T:8f}"):
            # dá para otimizar o cálculo das energias

            Eold = self.energies[-1]
            mag = self.magnetizations[-1]

            i = rnd.integers(0, self.L)
            j = rnd.integers(0, self.L)
            u = rnd.uniform(0,1)
            # índice a ser alterado

            self.flip(i, j)
            Erand = self.calc_ener()

            delta = bool(np.abs(Eold - Erand) > 0)

            chance = bool(np.exp(-1 * delta / (kb * self.T)) > u)
            
            if delta or chance:
                self.energies.append(Erand)
                self.magnetizations.append( mag + 2 * self.spins(i, j) )
            else:
                self.flip(i, j)
                self.energies.append(Eold)

    def neighbours(self, pos):
        X, Y = pos

        viz = [
            [X - 1, Y],         # cima
            [X + 1, Y],         # baixo
            [X, Y - 1],         # esquerda
            [X, Y + 1],         # direita
        ]

        # Condições periódicas

        for position in viz:
            x, y = position[0], position[1]
            if x == self.L:
                x = 0
            elif x == -1:
                x = self.L - 1
            if y == self.L:
                y = 0
            elif y == -1:
                y = self.L - 1
        
        to_return = np.isin(self.spins, viz)

        # retorna uma matriz com as mesmas dimensões de self.spins mas com booleanos com base em se estão em viz ou não
        
        return to_return

    @property
    def energy(self):
        # usa para aceder ao array com as energias
        return np.array(self.energies)

    @property
    def magnetization(self):
        # usa para aceder ao array com as magnetizações
        return np.array(self.magnetizations)

In [101]:
ising = IsingModel(32, 4)
ising.iter_monte_carlo(100)

energy = ising.energy
magnetization = ising.magnetization

X = np.array([a for a in range(0, len(energy))])
plt.plot(X, energy)
#plt.plot(X, magnetization)
plt.grid(True, ls="--")

L=    32, T=4.000000:   0%|          | 0/100 [00:00<?, ?it/s]


TypeError: 'numpy.ndarray' object is not callable